# NB39 - trimmed trailing scores on the full archive, in event time

NB38 screened trimmed trailing scores against forward Sharpe on the dense-polling regime only
(from 2026-04-01): about four months, 70 overlapping decisions. This notebook runs the same
question on the whole archive from mid-2025, which means running it on WEEKLY data for most of
its length: through 2025 the archive holds about one price-changing mark per vault per week,
one every two days in January-March 2026, and fifteen or more a day from April (cell 2).

Forward-filling weekly marks to a daily grid and then computing daily statistics is where the
gotchas live, so nothing here is computed on a daily grid. Every score uses observed marks
only: event log returns between consecutive marks, trimming by a FRACTION of events (so a weekly
vault loses one or three of its thirteen marks in 90 days and a daily vault loses nine or
twenty-two of ninety), staleness measured in days since the vault's own last mark, and forward
outcomes that must contain marks near the end of the window. The primary horizon is 60 days
because a 30-day window holds about four weekly marks; the 30-day horizon is kept as a
secondary target. The three polling regimes are screened together and separately.

**Focus is forward Sharpe.** Verdict DIAGNOSTIC: a screen, not a result. No vault is selected,
masked or tuned by name. **Headline: on 192 decisions over a year, trailing quality
DOES persist into the next 60 days - 15 of 16 signals clear the family-wise bound, led by
the 180-day Sharpe and Sortino at rho 0.26 - and trimming adds nothing to that under
a family bound.** The four-month screen of NB38 could not see this because its horizon was 30
days and its sample a third of the size.

**Based on:** [38-research-trimmed-return-screen.ipynb](38-research-trimmed-return-screen.ipynb)
(machinery and its two reviews), [33-research-lead-comparison.ipynb](33-research-lead-comparison.ipynb)
(the archive density table that defines the regimes). Snapshot `vault-prices.parquet`
255,548,076 bytes, sha256 `11e7c5e0103e1012`, last mark 2026-09-16 (cell 2).

## Method

Marks: one per vault per UTC day (the last poll of the day). Events: consecutive marks; event
return = log price ratio; event span = days between them. A candidate at decision T needs, in
the trailing window (T-1-W, T-1] for W in 90 and 180 days, at least 8 marks, its last mark within
14 days of T-1, and a TVL of at least 7,500 USD at that mark. Scores per window: return score
(sum of event returns, annualised over W; raw and with the best 10% and 25% of events removed),
Sharpe score (return score over event volatility sqrt(sum r^2 / W x 365), raw and trimmed the
same way), Sortino, event volatility. Forward outcomes over (T, T + H] for H = 60 (primary) and
30: log return from the mark carried at T to the last mark in the window, event volatility,
event Sharpe, log max drawdown on the mark path; a window needs at least 6 (H = 60) or 4
(H = 30) marks and one within 14 days of its end. Panel: 29,912 candidate-dates, 397
vaults, 192 decisions 2025-07-01 to 2026-07-18; 59,991 candidate-dates dropped as
stale or under-marked, 25,200 for TVL, 865 for an unobserved forward window (cell 4).
Marks per 90-day window, median: 13 in the weekly regime, 32 in the
transition, 90 in the dense regime; marks per 60-day forward window 9 /
60 / 60 (cell 4).

Inference as NB38: per-date signed Spearman averaged over dates, one two-way cluster bootstrap
(15-decision circular date blocks x vault clusters, 500 draws, seed 20260917) shared
across every hypothesis, studentised max-T simultaneous lower bounds over the 16-signal family on
the primary target (critical 2.21), paired trimmed-minus-raw differences on the
same draws with their own family bound, and a foresight-oracle reachability assertion (lower
bound 0.997, cell 8).

## Key new insights and what did we learn from this experiment?

**1. On a year of data, trailing quality persists into the next 60 days.** 15 of 16 signals
clear the simultaneous lower bound of zero on forward 60-day Sharpe; the only one that does not
is 180-day volatility (0.112, bound -0.010). The strongest are the 180-day
Sharpe and Sortino scores: `sharpe180_f10` 0.258 (bound 0.151),
`sortino180` 0.260 (bound 0.142), `sharpe180_f00` 0.254
(bound 0.137); the raw 90-day return score is the weakest that still clears,
0.152 (bound 0.044) (cell 6). The same scores predict forward RETURN at
0.196 to 0.200 and 30-day forward Sharpe at 0.214 to
0.225. NB38 saw 0.136 for the 180-day Sharpe on four months at a 30-day horizon and
could not clear a family bound; this screen has three times the decisions, a horizon that holds
enough marks, and a 2025 regime in which persistence is strongest.

**2. Trimming adds nothing that survives a family bound, and trimming a Sharpe score can hurt.**
The return-score trim at 90 days lifts the correlation from 0.152 to
0.211 (paired difference 0.059 [-0.004,
0.122], add-one p 0.084 alone, family bound
-0.015); at 180 days the difference is 0.006. As in NB38
the trimmed return score takes on a volatility loading (signed correlation with lower forward
volatility 0.133 raw, 0.500 trimmed) and lands where the raw 180-day
scores already are. Trimming the Sharpe score is flat at 10% (-0.012,
0.004) and negative at 25% (-0.037,
-0.035); among old vaults the 90-day 25% trim costs
-0.171 [-0.325, -0.041] (cell 10). A
Sharpe score already normalises by the size of its jumps; removing them takes information out.

**3. Persistence is strongest in the weekly regime and weakest in the dense one.** Weekly 2025
(92 decisions, 9,387 rows): `sharpe180_f00` 0.289 (bound
0.133), `sortino180` 0.300, every score but `vol180` clears. Transition
(45 decisions): `sharpe180_f10` 0.280 (bound 0.090), most
others just above or just below zero. Dense April on (55 decisions): `sharpe180_f00`
0.211 with bound -0.005 - the NB38 picture, a shade below the line
(cell 10). Two readings are consistent with this and the screen cannot separate them: quality
persisted more in 2025's universe (which was 99.2% under a year old), or weekly marks
smooth the outcome so that trailing and forward scores share the same smoothing. The event-time
construction removes the daily-grid artefact but not the fact that a weekly mark is itself a
week's average.

**4. The young cohort carries it.** Young vaults (75.8% of rows, 192 decisions):
`sharpe180_f00` 0.267 (bound 0.134), `ret90_f10`
0.225 (bound 0.123). Old vaults (102 decisions):
`sharpe180_f00` 0.200 (bound 0.008), `sortino180`
0.204, the return scores below the line (cell 10). The incumbent's 360-day CAGR leg
cannot score a young vault at all; the scores that predict best here need 180 days.

**5. What this says about the incumbent's ranker.** Its Sortino leg looks back 45 days and its
CAGR leg 360; this screen has no 45-day window, so the leg itself is not tested, but the pattern
is that 180-day risk-adjusted scores (0.254-0.260) predict better
than 90-day ones (0.204-0.214), and better than raw return
scores at either length. That is a lead for a ranker test, not a result: the effect on a
six-name book is what the standing gates measure, and portfolio consequences are not claimed
here.

## Summary of results

Forward 60-day Sharpe screen, all regimes (cell 6): signed Spearman, simultaneous lower bound
over the 16-signal family (critical 2.21), unadjusted add-one p; the forward volatility
column is signed so positive = the score's good end had LOWER forward volatility.

| signal | rho fwd60 Sharpe | lower bound | p | rho fwd60 return | rho fwd60 vol | rho fwd30 Sharpe |
|---|---|---|---|---|---|---|
| ret90_f00 | 0.152 | 0.044 | 0.004 | 0.138 | 0.133 | 0.109 |
| ret90_f10 | 0.211 | 0.112 | 0.002 | 0.200 | 0.500 | 0.177 |
| ret90_f25 | 0.207 | 0.105 | 0.002 | 0.202 | 0.595 | 0.176 |
| sharpe90_f00 | 0.214 | 0.109 | 0.002 | 0.156 | 0.152 | 0.172 |
| sharpe90_f10 | 0.202 | 0.099 | 0.002 | 0.149 | 0.174 | 0.169 |
| sharpe90_f25 | 0.177 | 0.064 | 0.004 | 0.139 | 0.260 | 0.153 |
| sortino90 | 0.204 | 0.100 | 0.002 | 0.151 | 0.159 | 0.158 |
| vol90 | 0.159 | 0.059 | 0.002 | 0.165 | 0.673 | 0.141 |
| ret180_f00 | 0.207 | 0.087 | 0.002 | 0.185 | 0.265 | 0.165 |
| ret180_f10 | 0.213 | 0.093 | 0.002 | 0.200 | 0.519 | 0.180 |
| ret180_f25 | 0.197 | 0.075 | 0.002 | 0.190 | 0.594 | 0.165 |
| sharpe180_f00 | 0.254 | 0.137 | 0.002 | 0.196 | 0.263 | 0.214 |
| sharpe180_f10 | 0.258 | 0.151 | 0.002 | 0.194 | 0.285 | 0.225 |
| sharpe180_f25 | 0.219 | 0.107 | 0.002 | 0.180 | 0.351 | 0.183 |
| sortino180 | 0.260 | 0.142 | 0.002 | 0.196 | 0.275 | 0.214 |
| vol180 | 0.112 | -0.010 | 0.016 | 0.123 | 0.608 | 0.099 |

Paired trimmed-minus-raw on forward 60-day Sharpe (cell 6): per-comparison 95% intervals and
add-one p; simultaneous lower bounds over each 8-comparison family are all below zero.

| | 10% of events removed | 25% of events removed |
|---|---|---|
| ret score, 90 d | 0.059 [-0.004, 0.122] p 0.084 | 0.055 [-0.024, 0.137] p 0.180 |
| ret score, 180 d | 0.006 [-0.043, 0.063] p 0.782 | -0.010 [-0.080, 0.067] p 0.830 |
| sharpe score, 90 d | -0.012 [-0.060, 0.038] p 0.539 | -0.037 [-0.109, 0.029] p 0.275 |
| sharpe score, 180 d | 0.004 [-0.049, 0.055] p 0.922 | -0.035 [-0.120, 0.051] p 0.391 |

Per regime and cohort, `sharpe180_f00` on forward 60-day Sharpe (cell 10): weekly 2025
0.289 (bound 0.133), transition 0.234 (0.037),
dense 0.211 (-0.005), young 0.267 (0.134),
old 0.200 (0.008).

## Robustness of results

- Nothing is computed on a daily grid: scores and outcomes are event-time on observed marks,
  eligibility needs a mark within 14 days of T-1 and 8 marks in the window, forward windows
  need 6 (60 d) or 4 (30 d) marks and one within 14 days of the end (cell 4). Inside those
  rules a weekly mark still averages a week; the forward outcome of a weekly vault is a
  60-day return over about nine marks, and its "Sharpe" is a coarse quantity.
- The screen is reachable: a foresight oracle clears the 17-signal bound at
  0.997 (cell 8).
- One bootstrap per screen; the regime and cohort screens are separate samples with separate
  bootstraps and their intervals are not comparable across screens in a paired sense.
- 192 decisions two days apart over about a year, each with a 60-day forward window:
  roughly six non-overlapping horizons; the 15-decision date blocks (30 days) are shorter than
  the horizon, so the bootstrap understates dependence between neighbouring blocks somewhat.
  The simultaneous bounds are still the controlled figures.
- Trimmed scores are ranking transformations, not investable returns; forward max drawdown is
  in log units.
- The panel is the archive, not the incumbent's candidate pool; no portfolio claim is made.


## Part 0. Archive, provenance, constants, regimes


In [1]:
import hashlib, json, math
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 200)

ARCHIVE = Path.home() / ".cache/tradingstrategy/vaults/downloads/vault-prices.parquet"
raw_bytes = ARCHIVE.read_bytes()
PROVENANCE = {"file": str(ARCHIVE), "bytes": len(raw_bytes), "sha256": hashlib.sha256(raw_bytes).hexdigest()}
del raw_bytes
HYPERCORE_CHAIN = 9999

WINDOWS = (90, 180)
TRIM_FRACTIONS = (0.0, 0.10, 0.25)
HORIZONS = {60: 6, 30: 4}          # forward horizon days -> minimum observed marks in the window
PRIMARY_H = 60
PANEL_START = pd.Timestamp("2025-07-01")
DECISION_STEP_DAYS = 2
MIN_TVL_USD = 7_500.0
MIN_EVENTS = 8
STALE_DAYS = 14
MIN_CANDIDATES = 8
MIN_DATES = 40
YOUNG_DAYS = 360
DRAWS = 500
DATE_BLOCK = 15
SEED = 20260917
LEVEL = 0.95
REGIMES = [("weekly 2025", pd.Timestamp("2025-01-01"), pd.Timestamp("2026-01-01")),
           ("transition Jan-Mar 2026", pd.Timestamp("2026-01-01"), pd.Timestamp("2026-04-01")),
           ("dense Apr 2026 on", pd.Timestamp("2026-04-01"), pd.Timestamp("2027-01-01"))]


def regime_of(t):
    for name, a, b in REGIMES:
        if a <= t < b:
            return name
    return "other"


df = pd.read_parquet(ARCHIVE, columns=["address", "chain", "share_price", "total_assets", "name"])
df = df[df["chain"] == HYPERCORE_CHAIN].reset_index()
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df[df["timestamp"] >= pd.Timestamp("2025-01-01")].sort_values(["address", "timestamp"])
df["date"] = df["timestamp"].dt.floor("D")
marks = df.groupby(["address", "date"])[["share_price", "total_assets"]].last().reset_index()
marks = marks[marks["share_price"] > 0]
names = df.groupby("address")["name"].last()
LAST_MARK = df["timestamp"].max()
display(pd.Series({**PROVENANCE, "last_mark": str(LAST_MARK), "hypercore_vaults": marks["address"].nunique(),
                   "mark_days": len(marks)}, name="value").to_frame())

# Polling density by month on the mark-day grid: marks per vault per day among vaults above the TVL floor.
mm = marks[marks["total_assets"] >= MIN_TVL_USD].copy()
mm["month"] = mm["date"].dt.to_period("M")
dens = mm.groupby("month").agg(vaults=("address", "nunique"), mark_days=("date", "size"))
dens["mark_days_per_vault_per_day"] = dens["mark_days"] / dens["vaults"] / 30.0
display(dens.round(3).T)


,value
file,/Users/moo/.cache/tradingstrategy/vaults/downl...
bytes,255548076
sha256,11e7c5e0103e10125a27fcb2ed58febd394cab3e317771...
last_mark,2026-09-16 07:25:06.705000
hypercore_vaults,604
mark_days,112919


month,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
vaults,57.000,65.000,73.000,81.000,92.000,101.000,133.000,153.000,172.000,197.000,211.000,230.000,257.000,300.000,341.000,322.00,332.000,320.00,297.000,304.000,271.000
mark_days,210.000,220.000,257.000,332.000,314.000,344.000,543.000,520.000,597.000,835.000,742.000,1023.000,2795.000,6831.000,8279.000,8307.00,8806.000,8157.00,8433.000,8091.000,4046.000
mark_days_per_vault_per_day,0.123,0.113,0.117,0.137,0.114,0.114,0.136,0.113,0.116,0.141,0.117,0.148,0.363,0.759,0.809,0.86,0.884,0.85,0.946,0.887,0.498


## Part 1. The event-time panel

One row per (decision date, vault). Nothing is forward-filled: a window's statistics come from
the marks inside it, and a vault whose last mark is older than 14 days is not a candidate.


In [2]:
def event_scores(r: np.ndarray, W: int) -> dict:
    """Raw and trimmed return and Sharpe scores from event returns `r` in a window of W days.
    Trimming removes the ceil(f x n) largest event returns; the window length stays the denominator,
    so these are ranking scores, not investable returns."""
    out = {}
    n = len(r)
    order = np.sort(r)
    for f in TRIM_FRACTIONS:
        k = int(math.ceil(f * n)) if f > 0 else 0
        kept = order[:n - k] if k else order
        rate = float(kept.sum() * 365.0 / W)
        vol = float(math.sqrt((kept ** 2).sum() * 365.0 / W)) if len(kept) else float("nan")
        tag = f"f{int(f * 100):02d}"
        out[f"ret_{tag}"] = rate
        out[f"sharpe_{tag}"] = rate / vol if vol and vol > 0 else float("nan")
    downside = math.sqrt((np.clip(r, None, 0.0) ** 2).sum() * 365.0 / W)
    out["sortino"] = out["ret_f00"] / downside if downside > 0 else float("nan")
    out["vol"] = float(math.sqrt((r ** 2).sum() * 365.0 / W))
    out["events"] = int(n)
    return out


def forward_outcome(carried: float, fwd_prices: np.ndarray, H: int) -> dict:
    """Outcome from the mark carried at T to the marks in (T, T + H]."""
    prices = np.concatenate([[carried], fwd_prices])
    r = np.diff(np.log(prices))
    total = float(r.sum())
    vol = float(math.sqrt((r ** 2).sum() * 365.0 / H))
    path = np.concatenate([[0.0], np.cumsum(r)])
    return {f"fwd{H}_return": total, f"fwd{H}_vol": vol,
            f"fwd{H}_sharpe": (total * 365.0 / H) / vol if vol > 0 else float("nan"),
            f"fwd{H}_log_max_dd": float(np.min(path - np.maximum.accumulate(path))),
            f"fwd{H}_events": int(len(fwd_prices))}


max_h = max(HORIZONS)
last_decision = LAST_MARK.floor("D") - pd.Timedelta(days=max_h)
decisions = pd.date_range(PANEL_START, last_decision, freq=f"{DECISION_STEP_DAYS}D")
rows = []
dropped = {"stale_or_too_few_marks": 0, "tvl": 0, "forward_marks": 0}
for address, g in marks.groupby("address"):
    g = g.set_index("date").sort_index()
    mdays = g.index
    prices = g["share_price"].to_numpy()
    tvls = g["total_assets"].to_numpy()
    born = mdays[0]
    for t in decisions:
        t1 = t - pd.Timedelta(days=1)
        i_last = int(mdays.searchsorted(t1, side="right")) - 1   # last mark at or before T-1
        if i_last < 0 or (t1 - mdays[i_last]).days > STALE_DAYS:
            dropped["stale_or_too_few_marks"] += 1
            continue
        if tvls[i_last] < MIN_TVL_USD:
            dropped["tvl"] += 1
            continue
        row = {"address": address, "date": t, "age_days": int((t - born).days), "regime": regime_of(t),
               "days_since_mark": int((t1 - mdays[i_last]).days)}
        scored_any = False
        for W in WINDOWS:
            start = t1 - pd.Timedelta(days=W)
            i_first = int(mdays.searchsorted(start, side="right"))   # first mark strictly after start
            n_marks = i_last - i_first + 1
            if n_marks < MIN_EVENTS or i_first == 0:
                for f in TRIM_FRACTIONS:
                    tag = f"f{int(f * 100):02d}"
                    row[f"ret{W}_{tag}"] = np.nan; row[f"sharpe{W}_{tag}"] = np.nan
                row[f"sortino{W}"] = np.nan; row[f"vol{W}"] = np.nan; row[f"events{W}"] = int(max(n_marks, 0))
                continue
            # Event returns between consecutive marks in the window; the first event uses the mark
            # just before the window as its start, so every mark in the window contributes one return.
            seg = prices[i_first - 1:i_last + 1]
            r = np.diff(np.log(seg))
            s = event_scores(r, W)
            for key, value in s.items():
                base, _, tag = key.partition("_")
                row[f"{base}{W}_{tag}" if tag else f"{base}{W}"] = value
            scored_any = True
        if not scored_any:
            dropped["stale_or_too_few_marks"] += 1
            continue
        i_carry = int(mdays.searchsorted(t, side="right")) - 1
        carried = float(prices[i_carry])
        ok_any = False
        for H, min_marks in HORIZONS.items():
            t_end = t + pd.Timedelta(days=H)
            j0 = int(mdays.searchsorted(t, side="right")); j1 = int(mdays.searchsorted(t_end, side="right"))
            fwd = prices[j0:j1]
            if len(fwd) < min_marks or (t_end - mdays[j1 - 1]).days > STALE_DAYS:
                for k in ("return", "vol", "sharpe", "log_max_dd"):
                    row[f"fwd{H}_{k}"] = np.nan
                row[f"fwd{H}_events"] = int(len(fwd))
                continue
            row.update(forward_outcome(carried, fwd, H))
            ok_any = True
        if not ok_any:
            dropped["forward_marks"] += 1
            continue
        rows.append(row)
panel = pd.DataFrame(rows)
panel["young"] = panel["age_days"] < YOUNG_DAYS
print("candidate-dates dropped:", dropped)
print(f"panel: {len(panel):,} rows, {panel['address'].nunique()} vaults, {panel['date'].nunique()} decisions "
      f"{panel['date'].min().date()} to {panel['date'].max().date()}")
by_regime = panel.groupby("regime").agg(rows=("address", "size"), vaults=("address", "nunique"), decisions=("date", "nunique"),
                                        events90_median=("events90", "median"), events180_median=("events180", "median"),
                                        fwd60_events_median=("fwd60_events", "median"), fwd30_events_median=("fwd30_events", "median"),
                                        fwd60_finite=("fwd60_sharpe", lambda s: float(np.isfinite(s).mean())),
                                        fwd30_finite=("fwd30_sharpe", lambda s: float(np.isfinite(s).mean())),
                                        young_share=("young", "mean"))
display(by_regime.round(3))

SIGNALS = []
for W in WINDOWS:
    for f in TRIM_FRACTIONS:
        tag = f"f{int(f * 100):02d}"
        SIGNALS.append({"name": f"ret{W}_{tag}", "direction": "high", "window": W, "trim": f, "family": "return"})
    for f in TRIM_FRACTIONS:
        tag = f"f{int(f * 100):02d}"
        SIGNALS.append({"name": f"sharpe{W}_{tag}", "direction": "high", "window": W, "trim": f, "family": "sharpe"})
    SIGNALS.append({"name": f"sortino{W}", "direction": "high", "window": W, "trim": None, "family": "sortino"})
    SIGNALS.append({"name": f"vol{W}", "direction": "low", "window": W, "trim": None, "family": "vol"})
SIGNAL_NAMES = [s["name"] for s in SIGNALS]
SIGNAL_SIGN = {s["name"]: (1.0 if s["direction"] == "high" else -1.0) for s in SIGNALS}
TARGETS = ["fwd60_sharpe", "fwd60_return", "fwd60_vol", "fwd60_log_max_dd", "fwd30_sharpe", "fwd30_return"]
TARGET_SIGN = {"fwd60_sharpe": 1.0, "fwd60_return": 1.0, "fwd60_vol": -1.0, "fwd60_log_max_dd": 1.0, "fwd30_sharpe": 1.0, "fwd30_return": 1.0}
PRIMARY = f"fwd{PRIMARY_H}_sharpe"
coverage = pd.DataFrame({s: np.isfinite(panel[s]).mean() for s in SIGNAL_NAMES}, index=["finite_share"]).T
display(coverage.round(3).T)


candidate-dates dropped: {'stale_or_too_few_marks': 59991, 'tvl': 25200, 'forward_marks': 865}
panel: 29,912 rows, 397 vaults, 192 decisions 2025-07-01 to 2026-07-18


,rows,vaults,decisions,events90_median,events180_median,fwd60_events_median,fwd30_events_median,fwd60_finite,fwd30_finite,young_share
regime,,,,,,,,,,
dense Apr 2026 on,12314,341,55,90.0,122.0,60.0,30.0,0.941,0.916,0.647
transition Jan-Mar 2026,8211,248,45,32.0,43.0,60.0,30.0,0.906,0.910,0.655
weekly 2025,9387,185,92,13.0,25.0,9.0,4.0,0.948,0.903,0.992


,ret90_f00,ret90_f10,ret90_f25,sharpe90_f00,sharpe90_f10,sharpe90_f25,sortino90,vol90,ret180_f00,ret180_f10,ret180_f25,sharpe180_f00,sharpe180_f10,sharpe180_f25,sortino180,vol180
finite_share,0.989,0.989,0.989,0.949,0.944,0.943,0.937,0.989,0.693,0.693,0.693,0.676,0.673,0.673,0.671,0.693


## Part 2. One shared bootstrap, every hypothesis

As NB38: per-date signed Spearman, equal-weight mean over dates, one two-way cluster bootstrap
shared by every signal, target and paired difference; simultaneous max-T lower bounds over the
signal family on the primary target and over the paired-comparison family.


In [3]:
def per_date_blocks(frame: pd.DataFrame) -> dict:
    out = {}
    cols = SIGNAL_NAMES + TARGETS
    for date, g in frame.groupby("date"):
        out[pd.Timestamp(date)] = {"vault": g["address"].to_numpy(), "values": g[cols].to_numpy(dtype=float)}
    return out


def date_statistics(values: np.ndarray) -> np.ndarray:
    S, T = len(SIGNAL_NAMES), len(TARGETS)
    out = np.full((S, T), np.nan)
    targets = values[:, S:]
    for i, name in enumerate(SIGNAL_NAMES):
        x = values[:, i]
        for j, target in enumerate(TARGETS):
            y = targets[:, j]
            ok = np.isfinite(x) & np.isfinite(y)
            if ok.sum() < MIN_CANDIDATES:
                continue
            xr, yr = rankdata(x[ok]), rankdata(y[ok])
            if np.ptp(xr) == 0 or np.ptp(yr) == 0:
                continue
            xc, yc = xr - xr.mean(), yr - yr.mean()
            out[i, j] = SIGNAL_SIGN[name] * TARGET_SIGN[target] * float((xc * yc).sum() / math.sqrt((xc ** 2).sum() * (yc ** 2).sum()))
    return out


def mean_over_dates(blocks: dict, dates: list, counts: dict | None = None) -> tuple:
    S, T = len(SIGNAL_NAMES), len(TARGETS)
    total, n = np.zeros((S, T)), np.zeros((S, T))
    for date in dates:
        block = blocks.get(date)
        if block is None:
            continue
        values = block["values"]
        if counts is not None:
            repeats = np.array([counts.get(v, 0) for v in block["vault"]], dtype=int)
            if repeats.sum() < MIN_CANDIDATES:
                continue
            values = np.repeat(values, repeats, axis=0)
        stats = date_statistics(values)
        finite = np.isfinite(stats)
        total[finite] += stats[finite]
        n[finite] += 1
    with np.errstate(invalid="ignore"):
        return np.where(n > 0, total / np.maximum(n, 1), np.nan), n


def bootstrap(frame: pd.DataFrame, draws: int = DRAWS, seed: int = SEED, verbose: bool = True) -> dict:
    blocks = per_date_blocks(frame)
    dates = sorted(blocks)
    vaults = sorted(frame["address"].unique())
    observed, n_dates = mean_over_dates(blocks, dates)
    rng = np.random.default_rng(seed)
    n_blocks = int(math.ceil(len(dates) / DATE_BLOCK))
    reps = np.full((draws,) + observed.shape, np.nan)
    for d in range(draws):
        starts = rng.integers(0, len(dates), size=n_blocks)
        index = np.concatenate([(np.arange(s, s + DATE_BLOCK) % len(dates)) for s in starts])[:len(dates)]
        drawn = rng.choice(len(vaults), size=len(vaults), replace=True)
        counts = {}
        for v in drawn:
            counts[vaults[v]] = counts.get(vaults[v], 0) + 1
        reps[d], _ = mean_over_dates(blocks, [dates[i] for i in index], counts)
        if verbose and (d + 1) % 100 == 0:
            print(f"  draw {d + 1}/{draws}")
    return {"observed": observed, "draws": reps, "n_dates": n_dates, "dates": dates, "rows": len(frame)}


def simultaneous_lower(observed: np.ndarray, reps: np.ndarray, level: float = LEVEL) -> dict:
    se = np.nanstd(reps, axis=0, ddof=1)
    member = np.isfinite(observed) & np.isfinite(se) & (se > 0)
    stud = (reps - observed[None, :]) / np.where(se > 0, se, np.nan)[None, :]
    complete = np.isfinite(stud[:, member]).all(axis=1) if member.any() else np.zeros(len(reps), dtype=bool)
    per_draw_max = stud[complete][:, member].max(axis=1) if complete.any() else np.array([])
    critical = float(np.percentile(per_draw_max, level * 100.0)) if len(per_draw_max) >= 100 else float("nan")
    lower = np.where(member, observed - critical * se, np.nan)
    centred = reps - observed[None, :]
    p = (1.0 + (centred >= observed[None, :]).sum(axis=0)) / (len(reps) + 1.0)
    return {"se": se, "critical": critical, "lower": lower, "lower_unadjusted": np.nanpercentile(reps, (1 - level) * 100.0, axis=0),
            "p": p, "family_size": int(member.sum()), "complete_draws": int(len(per_draw_max))}


def screen(frame: pd.DataFrame, label: str, verbose: bool = True) -> dict:
    print(f"{label}: {len(frame):,} rows, {frame['date'].nunique()} decisions, {frame['address'].nunique()} vaults")
    boot = bootstrap(frame, verbose=verbose)
    j = TARGETS.index(PRIMARY)
    fam = simultaneous_lower(boot["observed"][:, j], boot["draws"][:, :, j])
    rows = []
    for i, s in enumerate(SIGNALS):
        row = {"signal": s["name"], "family": s["family"], "window": s["window"], "trim": s["trim"],
               "dates": int(boot["n_dates"][i, j]), "evaluated": bool(boot["n_dates"][i, j] >= MIN_DATES and fam["se"][i] > 0)}
        for t_idx, target in enumerate(TARGETS):
            row[f"rho_{target}"] = boot["observed"][i, t_idx]
        row["se_primary"] = fam["se"][i]
        row["lo_primary_simultaneous"] = fam["lower"][i]
        row["lo_primary_unadjusted"] = fam["lower_unadjusted"][i]
        row["p_primary"] = fam["p"][i]
        rows.append(row)
    table = pd.DataFrame(rows).set_index("signal")
    diffs, d_obs, d_reps = [], [], []
    for W in WINDOWS:
        for fam_name in ("ret", "sharpe"):
            base = SIGNAL_NAMES.index(f"{fam_name}{W}_f00")
            for f in TRIM_FRACTIONS[1:]:
                idx = SIGNAL_NAMES.index(f"{fam_name}{W}_f{int(f * 100):02d}")
                obs = boot["observed"][idx, j] - boot["observed"][base, j]
                rep = boot["draws"][:, idx, j] - boot["draws"][:, base, j]
                d_obs.append(obs); d_reps.append(rep)
                fin = rep[np.isfinite(rep)]; n = len(fin); centred = fin - obs
                p_hi = (1.0 + (centred >= obs).sum()) / (n + 1.0); p_lo = (1.0 + (centred <= obs).sum()) / (n + 1.0)
                diffs.append({"family": fam_name, "window": W, "trim": f, "trimmed_rho": boot["observed"][idx, j],
                              "raw_rho": boot["observed"][base, j], "difference": obs,
                              "ci_lo": float(np.percentile(fin, 2.5)) if n >= 100 else np.nan,
                              "ci_hi": float(np.percentile(fin, 97.5)) if n >= 100 else np.nan,
                              "p_two_sided_add_one": float(min(1.0, 2 * min(p_hi, p_lo))) if n >= 100 else np.nan, "draws": int(n)})
    paired = pd.DataFrame(diffs)
    pfam = simultaneous_lower(np.array(d_obs), np.column_stack(d_reps))
    paired["lo_simultaneous_family"] = pfam["lower"]
    paired["se"] = pfam["se"]
    return {"label": label, "table": table, "paired": paired, "critical": fam["critical"], "family_size": fam["family_size"],
            "complete_draws": fam["complete_draws"], "boot": boot, "paired_critical": pfam["critical"], "paired_family_size": pfam["family_size"]}


TABLE_COLS = ["family", "window", "trim", "dates", "evaluated", f"rho_{PRIMARY}", "se_primary", "lo_primary_simultaneous",
              "lo_primary_unadjusted", "p_primary", "rho_fwd60_return", "rho_fwd60_vol", "rho_fwd60_log_max_dd", "rho_fwd30_sharpe", "rho_fwd30_return"]
full = screen(panel, "all regimes")
print(f"\nprimary family: {full['family_size']} signals, critical {full['critical']:.4f} on {full['complete_draws']} complete draws")
display(full["table"][TABLE_COLS].round(4))
print(f"\nPAIRED trimmed - raw on {PRIMARY} (per-comparison 95% intervals; simultaneous lower bound over the "
      f"{full['paired_family_size']} paired comparisons, critical {full['paired_critical']:.4f}):")
display(full["paired"].round(4))


all regimes: 29,912 rows, 192 decisions, 397 vaults


  draw 100/500


  draw 200/500


  draw 300/500


  draw 400/500


  draw 500/500

primary family: 16 signals, critical 2.2113 on 500 complete draws


,family,window,trim,dates,evaluated,rho_fwd60_sharpe,se_primary,lo_primary_simultaneous,lo_primary_unadjusted,p_primary,rho_fwd60_return,rho_fwd60_vol,rho_fwd60_log_max_dd,rho_fwd30_sharpe,rho_fwd30_return
signal,,,,,,,,,,,,,,,
ret90_f00,return,90,0.00,192,True,0.1523,0.0489,0.0442,0.0667,0.004,0.1382,0.1334,0.1498,0.1087,0.0935
ret90_f10,return,90,0.10,192,True,0.2109,0.0448,0.1119,0.1372,0.002,0.2003,0.5004,0.4884,0.1768,0.1596
ret90_f25,return,90,0.25,192,True,0.2071,0.0463,0.1047,0.1313,0.002,0.2020,0.5953,0.5700,0.1764,0.1596
sharpe90_f00,sharpe,90,0.00,192,True,0.2141,0.0477,0.1087,0.1343,0.002,0.1564,0.1524,0.1748,0.1725,0.1180
sharpe90_f10,sharpe,90,0.10,192,True,0.2024,0.0469,0.0987,0.1229,0.002,0.1486,0.1738,0.2076,0.1693,0.1174
sharpe90_f25,sharpe,90,0.25,192,True,0.1767,0.0511,0.0638,0.0911,0.004,0.1391,0.2600,0.2956,0.1527,0.1162
sortino90,sortino,90,NaN,192,True,0.2039,0.0468,0.1004,0.1266,0.002,0.1514,0.1591,0.1788,0.1578,0.1099
vol90,vol,90,NaN,192,True,0.1593,0.0453,0.0591,0.0842,0.002,0.1654,0.6730,0.6059,0.1407,0.1286
ret180_f00,return,180,0.00,192,True,0.2073,0.0546,0.0865,0.1158,0.002,0.1854,0.2650,0.2678,0.1646,0.1435



PAIRED trimmed - raw on fwd60_sharpe (per-comparison 95% intervals; simultaneous lower bound over the 8 paired comparisons, critical 2.2991):


,family,window,trim,trimmed_rho,raw_rho,difference,ci_lo,ci_hi,p_two_sided_add_one,draws,lo_simultaneous_family,se
0,ret,90,0.10,0.2109,0.1523,0.0586,-0.0041,0.1221,0.0838,500,-0.0154,0.0322
1,ret,90,0.25,0.2071,0.1523,0.0547,-0.0237,0.1370,0.1796,500,-0.0398,0.0411
2,sharpe,90,0.10,0.2024,0.2141,-0.0118,-0.0602,0.0377,0.5389,500,-0.0672,0.0241
3,sharpe,90,0.25,0.1767,0.2141,-0.0374,-0.1094,0.0285,0.2754,500,-0.1196,0.0358
4,ret,180,0.10,0.2131,0.2073,0.0058,-0.0428,0.0625,0.7824,500,-0.0578,0.0277
5,ret,180,0.25,0.1975,0.2073,-0.0098,-0.0801,0.0672,0.8303,500,-0.0945,0.0368
6,sharpe,180,0.10,0.2581,0.2537,0.0044,-0.0487,0.0551,0.9222,500,-0.0570,0.0267
7,sharpe,180,0.25,0.2191,0.2537,-0.0346,-0.1201,0.0513,0.3912,500,-0.1354,0.0438


### Reachability

A noisy foresight oracle (the primary target plus 5% noise) through the identical machinery must
clear the family-wise lower bound; otherwise an all-fail result says nothing about the signals.


In [4]:
rng = np.random.default_rng(SEED + 1)
oracle_panel = panel.copy()
oracle_panel["oracle"] = oracle_panel[PRIMARY] + rng.normal(0.0, 0.05 * float(np.nanstd(oracle_panel[PRIMARY])), len(oracle_panel))
_saved = (SIGNALS, SIGNAL_NAMES, SIGNAL_SIGN)
SIGNALS = list(SIGNALS) + [{"name": "oracle", "direction": "high", "window": None, "trim": None, "family": "oracle"}]
SIGNAL_NAMES = [s["name"] for s in SIGNALS]
SIGNAL_SIGN = {s["name"]: (1.0 if s["direction"] == "high" else -1.0) for s in SIGNALS}
try:
    oracle_res = screen(oracle_panel, "oracle reachability", verbose=False)
finally:
    SIGNALS, SIGNAL_NAMES, SIGNAL_SIGN = _saved
orow = oracle_res["table"].loc["oracle"]
print(f"oracle: rho {orow[f'rho_{PRIMARY}']:.4f}, simultaneous lower bound {orow['lo_primary_simultaneous']:.4f} over "
      f"{oracle_res['family_size']} signals (critical {oracle_res['critical']:.4f}); finite primary target rows "
      f"{int(np.isfinite(panel[PRIMARY]).sum())} of {len(panel)}")
assert orow["lo_primary_simultaneous"] > 0, "the screen cannot produce a positive simultaneous bound even for a foresight oracle"
print("reachable")


oracle reachability: 29,912 rows, 192 decisions, 397 vaults


oracle: rho 0.9975, simultaneous lower bound 0.9966 over 17 signals (critical 2.2113); finite primary target rows 27932 of 29912
reachable


## Part 3. Per regime, and young against old

The three polling regimes are separate samples with separate bootstraps; the weekly regime is
the one this notebook exists for. Young (< 360 days) and old are split on the whole panel.


In [5]:
SHORT_COLS = ["window", "trim", "dates", "evaluated", f"rho_{PRIMARY}", "lo_primary_simultaneous", "p_primary",
              "rho_fwd60_return", "rho_fwd60_vol", "rho_fwd30_sharpe"]
by_regime_screens = {}
for name, a, b in REGIMES:
    sub = panel[panel["regime"] == name]
    if sub["date"].nunique() < MIN_DATES:
        print(f"{name}: only {sub['date'].nunique()} decisions - not screened")
        continue
    res = screen(sub, name, verbose=False)
    by_regime_screens[name] = res
    print(f"  family {res['family_size']}, critical {res['critical']:.4f}, complete draws {res['complete_draws']}")
    display(res["table"][SHORT_COLS].round(4))
    print(f"  paired trimmed - raw on {PRIMARY} (family critical {res['paired_critical']:.4f}):")
    display(res["paired"].round(4))
young = screen(panel[panel["young"]], "young (< 360 days)", verbose=False)
old = screen(panel[~panel["young"]], "old (>= 360 days)", verbose=False)
for res in (young, old):
    print(f"\n{res['label']}: family {res['family_size']}, critical {res['critical']:.4f}")
    display(res["table"][SHORT_COLS].round(4))
    display(res["paired"].round(4))


weekly 2025: 9,387 rows, 92 decisions, 185 vaults


  family 16, critical 2.2913, complete draws 500


,window,trim,dates,evaluated,rho_fwd60_sharpe,lo_primary_simultaneous,p_primary,rho_fwd60_return,rho_fwd60_vol,rho_fwd30_sharpe
signal,,,,,,,,,,
ret90_f00,90,0.00,92,True,0.1929,0.0257,0.0020,0.1902,0.0808,0.1373
ret90_f10,90,0.10,92,True,0.2606,0.1220,0.0020,0.2656,0.3861,0.2103
ret90_f25,90,0.25,92,True,0.2592,0.1152,0.0020,0.2678,0.5108,0.2225
sharpe90_f00,90,0.00,92,True,0.2648,0.1230,0.0020,0.2303,0.1855,0.2218
sharpe90_f10,90,0.10,92,True,0.2705,0.1368,0.0020,0.2195,0.1406,0.2314
sharpe90_f25,90,0.25,92,True,0.2753,0.1405,0.0020,0.2324,0.1667,0.2379
sortino90,90,NaN,92,True,0.2478,0.1053,0.0020,0.2205,0.1841,0.1980
vol90,90,NaN,92,True,0.1761,0.0165,0.0040,0.1962,0.6636,0.1638
ret180_f00,180,0.00,92,True,0.2578,0.0993,0.0020,0.2541,0.2593,0.1828


  paired trimmed - raw on fwd60_sharpe (family critical 2.3022):


,family,window,trim,trimmed_rho,raw_rho,difference,ci_lo,ci_hi,p_two_sided_add_one,draws,lo_simultaneous_family,se
0,ret,90,0.10,0.2606,0.1929,0.0676,-0.0165,0.1558,0.1477,500,-0.0347,0.0444
1,ret,90,0.25,0.2592,0.1929,0.0663,-0.0507,0.1916,0.2715,500,-0.0714,0.0598
2,sharpe,90,0.10,0.2705,0.2648,0.0057,-0.0479,0.0629,0.7705,500,-0.0582,0.0277
3,sharpe,90,0.25,0.2753,0.2648,0.0105,-0.0593,0.0805,0.7146,500,-0.0719,0.0358
4,ret,180,0.10,0.2619,0.2578,0.0041,-0.0696,0.0704,0.8064,500,-0.0806,0.0368
5,ret,180,0.25,0.2423,0.2578,-0.0155,-0.1318,0.0876,0.8343,500,-0.1469,0.0571
6,sharpe,180,0.10,0.3040,0.2890,0.0150,-0.0499,0.0746,0.5948,500,-0.0579,0.0317
7,sharpe,180,0.25,0.2808,0.2890,-0.0081,-0.1161,0.0920,0.9541,500,-0.1280,0.0520


...

,window,trim,dates,evaluated,rho_fwd60_sharpe,lo_primary_simultaneous,p_primary,rho_fwd60_return,rho_fwd60_vol,rho_fwd30_sharpe
signal,,,,,,,,,,
ret90_f00,90,0.00,192,True,0.1478,0.0302,0.004,0.1281,0.1298,0.0996
ret90_f10,90,0.10,192,True,0.2247,0.1229,0.002,0.2032,0.4858,0.1872
ret90_f25,90,0.25,192,True,0.2244,0.1183,0.002,0.2087,0.5803,0.1902
sharpe90_f00,90,0.00,192,True,0.2169,0.1067,0.002,0.1536,0.1555,0.1672
sharpe90_f10,90,0.10,192,True,0.2146,0.1097,0.002,0.1528,0.1739,0.1726
sharpe90_f25,90,0.25,192,True,0.2005,0.0911,0.002,0.1544,0.2564,0.1595
sortino90,90,NaN,192,True,0.2063,0.0960,0.002,0.1483,0.1610,0.1516
vol90,90,NaN,192,True,0.1766,0.0689,0.002,0.1716,0.6556,0.1599
ret180_f00,180,0.00,192,True,0.2244,0.0884,0.002,0.1914,0.2688,0.1732


,family,window,trim,trimmed_rho,raw_rho,difference,ci_lo,ci_hi,p_two_sided_add_one,draws,lo_simultaneous_family,se
0,ret,90,0.10,0.2247,0.1478,0.0769,0.0126,0.1504,0.0439,500,-0.0088,0.0354
1,ret,90,0.25,0.2244,0.1478,0.0766,-0.0098,0.1593,0.0838,500,-0.0307,0.0443
2,sharpe,90,0.10,0.2146,0.2169,-0.0023,-0.0520,0.0514,0.8064,500,-0.0648,0.0258
3,sharpe,90,0.25,0.2005,0.2169,-0.0164,-0.0952,0.0501,0.6108,500,-0.1128,0.0398
4,ret,180,0.10,0.2393,0.2244,0.0150,-0.0443,0.0779,0.6267,500,-0.0600,0.0309
5,ret,180,0.25,0.2274,0.2244,0.0031,-0.0757,0.0800,0.9541,500,-0.0957,0.0408
6,sharpe,180,0.10,0.2683,0.2674,0.0010,-0.0607,0.0648,0.8583,500,-0.0751,0.0314
7,sharpe,180,0.25,0.2464,0.2674,-0.0210,-0.1077,0.0764,0.5948,500,-0.1391,0.0487



old (>= 360 days): family 16, critical 2.2817


,window,trim,dates,evaluated,rho_fwd60_sharpe,lo_primary_simultaneous,p_primary,rho_fwd60_return,rho_fwd60_vol,rho_fwd30_sharpe
signal,,,,,,,,,,
ret90_f00,90,0.00,102,True,0.1333,-0.0398,0.0379,0.1352,0.1970,0.1094
ret90_f10,90,0.10,102,True,0.1169,-0.0746,0.0758,0.1420,0.6521,0.0994
ret90_f25,90,0.25,102,True,0.0906,-0.1005,0.1337,0.1270,0.7207,0.0702
sharpe90_f00,90,0.00,102,True,0.1627,-0.0037,0.0100,0.1018,0.1017,0.1450
sharpe90_f10,90,0.10,102,True,0.0988,-0.0823,0.0838,0.0664,0.2024,0.0998
sharpe90_f25,90,0.25,102,True,-0.0084,-0.1975,0.5469,-0.0111,0.3590,0.0410
sortino90,90,NaN,102,True,0.1567,-0.0122,0.0140,0.1023,0.1239,0.1375
vol90,90,NaN,102,True,0.0882,-0.0961,0.1477,0.1315,0.7382,0.0444
ret180_f00,180,0.00,102,True,0.1276,-0.0682,0.0519,0.1235,0.2719,0.1205


,family,window,trim,trimmed_rho,raw_rho,difference,ci_lo,ci_hi,p_two_sided_add_one,draws,lo_simultaneous_family,se
0,ret,90,0.10,0.1169,0.1333,-0.0165,-0.1437,0.1150,0.7864,500,-0.1587,0.0636
1,ret,90,0.25,0.0906,0.1333,-0.0428,-0.1863,0.1090,0.5389,500,-0.2088,0.0742
2,sharpe,90,0.10,0.0988,0.1627,-0.0639,-0.1815,0.0263,0.2395,500,-0.1787,0.0513
3,sharpe,90,0.25,-0.0084,0.1627,-0.1711,-0.3254,-0.0415,0.0359,500,-0.3410,0.0760
4,ret,180,0.10,0.1133,0.1276,-0.0143,-0.1308,0.0936,0.7385,500,-0.1408,0.0566
5,ret,180,0.25,0.0897,0.1276,-0.0379,-0.1710,0.0877,0.4990,500,-0.1851,0.0658
6,sharpe,180,0.10,0.1928,0.2003,-0.0075,-0.1168,0.0969,0.8343,500,-0.1299,0.0547
7,sharpe,180,0.25,0.1092,0.2003,-0.0911,-0.2767,0.0768,0.2954,500,-0.2891,0.0885


## Part 4. Manifest


In [6]:
def table_records(res):
    return {"table": res["table"].round(6).to_dict(orient="index"), "paired": res["paired"].round(6).to_dict(orient="records"),
            "critical": res["critical"], "family_size": res["family_size"], "complete_draws": res["complete_draws"],
            "paired_critical": res["paired_critical"], "paired_family_size": res["paired_family_size"],
            "rows": int(res["boot"]["rows"]), "decisions": int(len(res["boot"]["dates"]))}

manifest = {
    "verdict": "DIAGNOSTIC - a vault-level screen on the full archive, not a result",
    "provenance": {**PROVENANCE, "last_mark": str(LAST_MARK)},
    "constants": {"windows": list(WINDOWS), "trim_fractions": list(TRIM_FRACTIONS), "horizons": {str(k): v for k, v in HORIZONS.items()},
                  "primary": PRIMARY, "panel_start": str(PANEL_START.date()), "min_tvl_usd": MIN_TVL_USD, "min_events": MIN_EVENTS,
                  "stale_days": STALE_DAYS, "min_candidates": MIN_CANDIDATES, "min_dates": MIN_DATES, "young_days": YOUNG_DAYS,
                  "draws": DRAWS, "date_block": DATE_BLOCK, "seed": SEED},
    "regimes": [(n, str(a.date()), str(b.date())) for n, a, b in REGIMES],
    "density": dens.round(6).reset_index().astype({"month": str}).to_dict(orient="records"),
    "panel": {"rows": int(len(panel)), "vaults": int(panel["address"].nunique()), "decisions": int(panel["date"].nunique()),
              "first": str(panel["date"].min().date()), "last": str(panel["date"].max().date()), "young_share": float(panel["young"].mean())},
    "dropped": dropped,
    "by_regime": by_regime.round(6).to_dict(orient="index"),
    "coverage": coverage["finite_share"].round(6).to_dict(),
    "screens": {"all": table_records(full), **{n: table_records(r) for n, r in by_regime_screens.items()},
                "young": table_records(young), "old": table_records(old)},
    "oracle": {"rho": float(orow[f"rho_{PRIMARY}"]), "lo_simultaneous": float(orow["lo_primary_simultaneous"]),
               "family_size": oracle_res["family_size"], "critical": oracle_res["critical"]},
}
Path("_build/manifest_39.json").write_text(json.dumps(manifest, indent=1, default=str))
print("wrote _build/manifest_39.json")


wrote _build/manifest_39.json
